In [4]:
import os
import arcpy
import logging
from datetime import datetime

def setup_logging(workspace):
    """Setup logging configuration"""
    log_file = os.path.join(workspace, f"habitat_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
    
    # Add format with timestamp
    log_format = '%(asctime)s - %(levelname)s - %(message)s'
    
    # Configure logging with both file and console output
    logging.basicConfig(
        level=logging.INFO,
        format=log_format,
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    
    # Add an initial log message to verify logging is working
    logging.info("Logging initialized successfully")
    logging.info(f"Log file created at: {log_file}")

def validate_inputs(project, year, map_name="Map"):
    """Validate input project and map"""
    try:
        print(f"Looking for map: {map_name}")
        map_view = project.listMaps(map_name)[0]
        
        print(f"Looking for vegetation layer: veg_{year}_wfp_hf_ght_final")
        landscape_layer = map_view.listLayers(f"veg_{year}_wfp_hf_ght_final")[0]
        if not landscape_layer:
            raise ValueError(f"Could not find vegetation layer for year {year}")
            
        print("Looking for points layer: landscape_summary_camaru_pts_2010_2023")
        locations = map_view.listLayers("landscape_summary_camaru_pts_2010_2023")[0]
        if not locations:
            raise ValueError("Could not find points layer")
            
        return map_view, landscape_layer, locations
    except IndexError as e:
        print(f"ERROR: Required map or layers not found: {str(e)}")
        raise ValueError(f"Required map or layers not found: {str(e)}")

def get_disturbance_query(hypothesis):
    """Return the appropriate disturbance query based on hypothesis"""
    
    if hypothesis == 1:
        # Only non-fire, non-harvest polygonal disturbances
        return (
            "FEATURE_TY IN ('RIS-CLEARING-UNKNOWN', 'FACILITY-UNKNOWN', 'SUMP', 'BORROWPIT-WET', 'RECREATION', "
            "'CROP', 'WELL-OIL', 'MISC-OIL-GAS-FACILITY', 'WELL-UNKNOWN', 'WELL-CLEARED-NOT-DRILLED', "
            "'TAME_PASTURE', 'WELL-GAS', 'BORROWPIT-DRY', 'FACILITY-OTHER', 'CULTIVATION', "
            "'OIL-GAS-PLANT', 'BORROWPITS', 'CAMPGROUND', 'WELL-CASED', 'GRVL-SAND-PIT', "
            "'ROUGH_PASTURE', 'LAGOON', 'WELL-BITUMEN', 'DUGOUT', 'WELL-BIT', 'CLEARING-UNKNOWN', "
            "'CLEARING-WELLPAD-UNCONFIRMED', 'RURAL-RESIDENCE', 'WELL-ABAND', 'OPEN-PIT-MINE', 'CAMP-INDUSTRIAL', "
            "'WELL-OTHER')"
        )
    elif hypothesis == 2:
        # Non-fire, non-harvest polygonal disturbances + hard linear features
        return (
            "FEATURE_TY IN ('RIS-CLEARING-UNKNOWN', 'FACILITY-UNKNOWN', 'SUMP', 'BORROWPIT-WET', 'RECREATION', "
            "'CROP', 'WELL-OIL', 'MISC-OIL-GAS-FACILITY', 'WELL-UNKNOWN', 'WELL-CLEARED-NOT-DRILLED', "
            "'TAME_PASTURE', 'WELL-GAS', 'RLWY-SGL-TRACK', 'BORROWPIT-DRY', 'FACILITY-OTHER', 'CULTIVATION', "
            "'OIL-GAS-PLANT', 'BORROWPITS', 'CAMPGROUND', 'WELL-CASED', 'GRVL-SAND-PIT', "
            "'ROUGH_PASTURE', 'LAGOON', 'WELL-BITUMEN', 'DUGOUT', 'WELL-BIT', 'CLEARING-UNKNOWN', "
            "'CLEARING-WELLPAD-UNCONFIRMED', 'RURAL-RESIDENCE', 'WELL-ABAND', 'OPEN-PIT-MINE', 'CAMP-INDUSTRIAL', "
            "'WELL-OTHER', 'ROAD-GRAVEL-1L', 'ROAD-GRAVEL-2L', 'ROAD-PAVED-UNDIV-1L', 'ROAD-PAVED-UNDIV-2L', "
            "'TRANSMISSION-LINE', 'PIPELINE', 'TRUCK-TRAIL', 'ROAD-UNIMPROVED', 'ROAD-UNCLASSIFIED', "
            "'VEGETATED-EDGE-RAILWAYS', 'VEGETATED-EDGE-ROADS')"
        )
    elif hypothesis == 3:
        # All disturbances, including all linear features
        return (
            "FEATURE_TY IN ('RIS-CLEARING-UNKNOWN', 'FACILITY-UNKNOWN', 'SUMP', 'BORROWPIT-WET', 'RECREATION', "
            "'CROP', 'WELL-OIL', 'MISC-OIL-GAS-FACILITY', 'WELL-UNKNOWN', 'WELL-CLEARED-NOT-DRILLED', "
            "'TAME_PASTURE', 'WELL-GAS', 'RLWY-SGL-TRACK', 'BORROWPIT-DRY', 'FACILITY-OTHER', 'CULTIVATION', "
            "'OIL-GAS-PLANT', 'BORROWPITS', 'CAMPGROUND', 'PRE-LOW-IMPACT-SEISMIC', 'WELL-CASED', 'GRVL-SAND-PIT', "
            "'TRAIL', 'ROUGH_PASTURE', 'LAGOON', 'WELL-BITUMEN', 'DUGOUT', 'WELL-BIT', 'CLEARING-UNKNOWN', "
            "'CLEARING-WELLPAD-UNCONFIRMED', 'RURAL-RESIDENCE', 'WELL-ABAND', 'OPEN-PIT-MINE', 'CAMP-INDUSTRIAL', "
            "'WELL-OTHER', 'ROAD-GRAVEL-1L', 'ROAD-GRAVEL-2L', 'ROAD-PAVED-UNDIV-1L', 'ROAD-PAVED-UNDIV-2L', "
            "'TRANSMISSION-LINE', 'PIPELINE', 'TRUCK-TRAIL', 'ROAD-UNIMPROVED', 'ROAD-UNCLASSIFIED', "
            "'VEGETATED-EDGE-RAILWAYS', 'VEGETATED-EDGE-ROADS', "
            "'LOW-IMPACT-SEISMIC', 'CONVENTIONAL-SEISMIC')"
        )
    else:
        raise ValueError(f"Invalid hypothesis number: {hypothesis}")
        
def process_year(survey_year, hypothesis, workspace):
    """Process habitat analysis for a specific year and hypothesis"""
    print(f"\n{'='*50}")
    print(f"Processing year {survey_year} for hypothesis {hypothesis}")
    print(f"{'='*50}")
    
    temp_layers = []
    output_files = []
    
    def ensure_output_path(filepath):
        if arcpy.Exists(filepath):
            try:
                arcpy.Delete_management(filepath)
                logging.info(f"Deleted existing file: {filepath}")
            except arcpy.ExecuteError:
                raise RuntimeError(f"Unable to delete existing file: {filepath}. File may be locked.")
        output_files.append(filepath)
    
    try:
        # Validate inputs for specific year
        print("Validating inputs...")
        project = arcpy.mp.ArcGISProject("CURRENT")
        _, landscape_layer, locations_layer = validate_inputs(project, survey_year)
        print(f"Found landscape layer: {landscape_layer.name}")
        print(f"Found locations layer: {locations_layer.name}")

        # Step 3: Suitable Habitat Filtering
        print("\nStep 3: Filtering suitable habitat...")
        logging.info("Filtering suitable BTNW habitat...")
        suitable_habitat_query = (
            "Combined_v7 IN ('Decid', 'Mixedwood', 'Spruce') AND "
            "(Origin_Year = 9999 OR " # Natural old forest
            f"(Origin_Year <= {survey_year - 80}))" # Or forest older than 80 years
        )
        
        suitable_habitat_layer = os.path.join(workspace, f"suitable_habitat_layer_{survey_year}.shp")
        ensure_output_path(suitable_habitat_layer)
        
        arcpy.MakeFeatureLayer_management(landscape_layer, "habitat_layer")
        temp_layers.append("habitat_layer")

        print("\nChecking available fields in habitat layer...")
        field_names = [field.name for field in arcpy.ListFields("habitat_layer")]
        print("Available fields:", field_names)

        print("\nQuery being used:", suitable_habitat_query)
        arcpy.SelectLayerByAttribute_management("habitat_layer", "NEW_SELECTION", suitable_habitat_query)
        arcpy.CopyFeatures_management("habitat_layer", suitable_habitat_layer)
        
        # Step 4: Fire Filtering
        print("\nStep 4: Filtering fire data...")
        logging.info("Filtering fire data...")
        fire_query = (
            "Origin_Source_NatDist IN ('SRD_Fire', 'WBNP_Fire') AND "
            f"YEAR >= {survey_year - 80} AND " # Fires that happened in last 80 years
            f"YEAR <= {survey_year} AND " # Fires up to current year
            "Combined_v7 IN ('Decid', 'Mixedwood', 'Spruce')"
        )
        
        fire_refined = os.path.join(workspace, f"fire_filtered_{survey_year}.shp")
        ensure_output_path(fire_refined)
        
        arcpy.MakeFeatureLayer_management(landscape_layer, "fire_layer")
        temp_layers.append("fire_layer")
        arcpy.SelectLayerByAttribute_management("fire_layer", "NEW_SELECTION", fire_query)
        arcpy.CopyFeatures_management("fire_layer", fire_refined)
        
        # Step 5: Harvest Filtering
        print("\nStep 5: Filtering harvest data...")
        logging.info("Filtering harvest data...")
        harvest_query = (
            "FEATURE_TY = 'HARVEST-AREA' AND "
            f"YEAR >= {survey_year - 80} AND " # Harvests in last 80 years
            f"YEAR <= {survey_year}"  # Harvests up to current year
        )
        
        harvest_refined = os.path.join(workspace, f"harvest_filtered_{survey_year}.shp")
        ensure_output_path(harvest_refined)
        
        arcpy.MakeFeatureLayer_management(landscape_layer, "harvest_layer")
        temp_layers.append("harvest_layer")
        arcpy.SelectLayerByAttribute_management("harvest_layer", "NEW_SELECTION", harvest_query)
        arcpy.CopyFeatures_management("harvest_layer", harvest_refined)
        
        # Step 6: Hypothesis-Specific Disturbance Selection
        print(f"\nStep 6: Filtering disturbances for hypothesis {hypothesis}...")
        logging.info(f"Filtering disturbances for hypothesis {hypothesis}...")
        disturbances_query = get_disturbance_query(hypothesis)
        
        disturbances_refined = os.path.join(workspace, f"disturbances_refined_hyp{hypothesis}_{survey_year}.shp")
        ensure_output_path(disturbances_refined)
        
        arcpy.MakeFeatureLayer_management(landscape_layer, "disturbances_layer")
        temp_layers.append("disturbances_layer")
        arcpy.SelectLayerByAttribute_management("disturbances_layer", "NEW_SELECTION", disturbances_query)
        arcpy.CopyFeatures_management("disturbances_layer", disturbances_refined)
        
        # Step 7: Process Habitat Layers
        print("\nStep 7: Processing habitat layers...")
        logging.info("Processing habitat layers...")
        habitat_minus_disturbance = os.path.join(workspace, f"habitat_minus_disturbance_hyp{hypothesis}_{survey_year}.shp")
        habitat_minus_fire = os.path.join(workspace, f"habitat_minus_fire_hyp{hypothesis}_{survey_year}.shp")
        habitat_minus_harvest = os.path.join(workspace, f"habitat_minus_harvest_hyp{hypothesis}_{survey_year}.shp")
        dissolved_habitat = os.path.join(workspace, f"dissolved_habitat_hyp{hypothesis}_{survey_year}.shp")
        
        ensure_output_path(habitat_minus_disturbance)
        ensure_output_path(habitat_minus_fire)
        ensure_output_path(habitat_minus_harvest)
        ensure_output_path(dissolved_habitat)
        
        arcpy.analysis.Erase(suitable_habitat_layer, disturbances_refined, habitat_minus_disturbance)
        arcpy.analysis.Erase(habitat_minus_disturbance, fire_refined, habitat_minus_fire)
        arcpy.analysis.Erase(habitat_minus_fire, harvest_refined, habitat_minus_harvest)
        arcpy.Dissolve_management(habitat_minus_harvest, dissolved_habitat, multi_part="SINGLE_PART")
        
        # Step 8: Calculate Area
        logging.info("Calculating areas...")

        # First make feature layers for proper joining
        arcpy.MakeFeatureLayer_management(dissolved_habitat, "dissolved_habitat_layer")
        arcpy.MakeFeatureLayer_management(locations_layer, "locations_layer")

        # Add and calculate area field
        arcpy.AddField_management("dissolved_habitat_layer", "Area_sqm", "DOUBLE")
        arcpy.CalculateField_management(
            "dissolved_habitat_layer", 
            "Area_sqm", 
            "!shape.area!", 
            "PYTHON3"
        )

        # Select points for the specific year
        location_year_query = f"survey_year = {survey_year}"
        arcpy.SelectLayerByAttribute_management("locations_layer", "NEW_SELECTION", location_year_query)

        # Spatial join to get areas
        points_with_habitat_area = os.path.join(workspace, f"points_with_habitat_area_hyp{hypothesis}_{survey_year}.shp")
        ensure_output_path(points_with_habitat_area)

        # Use old-style spatial join for compatibility
        arcpy.SpatialJoin_analysis(
            target_features="locations_layer",
            join_features="dissolved_habitat_layer",
            out_feature_class=points_with_habitat_area,
            join_operation="JOIN_ONE_TO_ONE",
            join_type="KEEP_ALL",
            match_option="INTERSECT"
        )

        # Calculate distances to nearest habitat patch
        arcpy.AddField_management(points_with_habitat_area, "Dist_patch", "DOUBLE")
        arcpy.Near_analysis(points_with_habitat_area, "dissolved_habitat_layer")

        # Export results
        csv_output = os.path.join(workspace, f"patch_metrics_hyp{hypothesis}_{survey_year}.csv")
        arcpy.TableToTable_conversion(points_with_habitat_area, workspace, os.path.basename(csv_output))

        return True
    
    except arcpy.ExecuteError as e:
        logging.error(f"ArcPy error in process_year: {arcpy.GetMessages()}")
        print(f"\nArcPy error occurred while processing year {survey_year}, hypothesis {hypothesis}")
        return False
        
    except Exception as e:
        logging.error(f"Error in process_year: {str(e)}")
        print(f"\nError occurred while processing year {survey_year}, hypothesis {hypothesis}")
        return False
        
    finally:
        # Clean up temporary layers
        for layer in temp_layers:
            if arcpy.Exists(layer):
                try:
                    arcpy.Delete_management(layer)
                except:
                    logging.warning(f"Could not delete temporary layer: {layer}")
    

def main():
    workspace_base_dir = r"C:\Users\hartt\Documents\Chapter 2\Chapter 2 Analysis"
    workspace = os.path.join(workspace_base_dir, "Extract_patch_metrics", "0_data", "1_processed", "10000m_outputs")
    start_year = 2021
    end_year = 2023
    
    if not os.path.exists(workspace):
        os.makedirs(workspace)
    
    arcpy.env.workspace = workspace
    arcpy.env.overwriteOutput = True
    
    setup_logging(workspace)
    
    try:
        print("\nReady to process years from", start_year, "to", end_year)
        response = input("Press Enter to continue or 'q' to quit: ")
        if response.lower() == 'q':
            print("Script terminated by user")
            return
        
        for hypothesis in [1, 2, 3]:
            print(f"\n{'='*50}")
            print(f"Starting processing for hypothesis {hypothesis}")
            print(f"{'='*50}")
            logging.info(f"\nStarting processing for hypothesis {hypothesis}")
            
            for year in range(start_year, end_year + 1):
                print(f"\nProcessing year {year} for hypothesis {hypothesis}")
                logging.info(f"\nProcessing year {year} for hypothesis {hypothesis}")
                success = process_year(year, hypothesis, workspace)
                
                if not success:
                    response = input(f"\nFailed to process year {year} for hypothesis {hypothesis}. Continue? (y/n): ")
                    if response.lower() != 'y':
                        print("Script terminated by user")
                        return
    except arcpy.ExecuteError:
        logging.error(f"ArcPy error: {arcpy.GetMessages()}")
        print("\nAn ArcPy error occurred. Check the log file for details.")
        raise
                     
    except Exception as e:
        logging.error(f"Fatal error: {str(e)}")
        print("\nAn unexpected error occurred. Check the log file for details.")
        raise
        
    finally:
        print("\nProcessing complete. Check the log file for details.")

if __name__ == "__main__":
    main()

2024-11-15 13:37:51,817 - INFO - Logging initialized successfully
2024-11-15 13:37:51,819 - INFO - Log file created at: C:\Users\hartt\Documents\Chapter 2\Chapter 2 Analysis\Extract_patch_metrics\0_data\1_processed\10000m_outputs\habitat_analysis_20241115_133751.log



Ready to process years from 2021 to 2023
Press Enter to continue or 'q' to quit: 

Starting processing for hypothesis 1


2024-11-15 13:37:53,478 - INFO - 
Starting processing for hypothesis 1



Processing year 2021 for hypothesis 1


2024-11-15 13:37:53,479 - INFO - 
Processing year 2021 for hypothesis 1


...


Step 6: Filtering disturbances for hypothesis 3...


2024-11-15 14:00:33,807 - INFO - Filtering disturbances for hypothesis 3...



Step 7: Processing habitat layers...


2024-11-15 14:00:36,590 - INFO - Processing habitat layers...
2024-11-15 14:00:55,200 - INFO - Calculating areas...



Processing complete. Check the log file for details.
